In [2]:
# This code represnets an example of a ML workflow. It classifies whether a news is fake or real.

import nltk
import string
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer 
from functools import lru_cache  # for caching stemmed words
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.metrics import accuracy_score , precision_score 

ps = PorterStemmer()
Tf = TfidfVectorizer()

def ensure_nltk_data():
    resources = {
        "tokenizers/punkt_tab": "punkt_tab",
        "corpora/stopwords": "stopwords",
    }
    for path, name in resources.items():
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(name)

ensure_nltk_data()

stop_words = set(stopwords.words("english"))  # load once

@lru_cache(maxsize=None)
def stem_word(w):
    return ps.stem(w)  # each unique word is stemmed only once

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_colwidth', 100)

data = pd.read_csv("C:\\Users\\sulem\\OneDrive\\Desktop\\Codes\\ML\\Datasets\\News.csv", encoding="latin-1")

# print(data.sample(20))

x = data[["Headline"]]
y = data["Target"]

def TextTransformation(text):

    text = text.lower()
    text = nltk.word_tokenize(text)

    Text = []
    for i in text:
        if i.isalnum():
            Text.append(i)

    TEXT = []
    for j in Text:
        if j not in stop_words and j not in string.punctuation:  
            TEXT.append(j)

    Sentences = []
    for k in TEXT:
        Sentences.append(ps.stem(k))

    return " ".join(Sentences)

def text_series(text):
    return text.apply(TextTransformation)

T = FunctionTransformer(text_series)

z = make_column_transformer(
    (make_pipeline(T, Tf), "Headline"),
    remainder="drop"
)

trainx,testx,trainy,testy = train_test_split(x,y, test_size=0.3, random_state=33)

m = LogisticRegression(class_weight="balanced") 

# By using Logistic Regression every Fake example in the loss function gets counted as if it were ~2.6x more important than a Real example. 
# The optimizer now gets punished much harder for misclassifying Fake headlines, so it stops defaulting to "Real" just because Real is more common.

pipe = make_pipeline(z,m)

pipe.fit(trainx,trainy)
predy = pipe.predict(testx)

print("The Accuracy score after Logistic Regression is:",accuracy_score(testy,predy))
print("The Precision Score after Logistic Regression is:",precision_score(testy,predy))

print("-------------------------------------------------------------------------------------")

new_data = pd.DataFrame({
    "Headline": ["Hedge Funds Are Warming Up To Leggett & Platt, Inc. (LEG)"]
})

prediction = pipe.predict(new_data)
result = "Fake" if prediction[0] == 0 else "Real"
print("The News is",result)

The Accuracy score after Logistic Regression is: 0.8467928902627512
The Precision Score after Logistic Regression is: 0.9148390771859869
-------------------------------------------------------------------------------------
The News is Fake
